# 📓 Notebook 03 – Training & Evaluation
**Version 1 | Road Damage Detection**

This notebook covers:
1. Train MobileNetV2
2. Train Custom CNN
3. Fine-tune YOLOv8n
4. Evaluate all models — Accuracy, Precision, Recall, F1
5. Plot training curves and confusion matrix
6. Side-by-side model comparison

In [ ]:
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path('..') / 'src'))

from dataset  import get_dataloaders
from models   import get_model, NUM_CLASSES
from train    import train_model
from evaluate import evaluate_model, plot_training_history

DATA_ROOT  = Path('../data/prepared').resolve()
MODELS_DIR = Path('../models').resolve()
PLOTS_DIR  = Path('../outputs/plots').resolve()
MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device    : {DEVICE}')
print(f'Data root : {DATA_ROOT}')

Device    : cpu
Data root : C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\v1\data\prepared


## 1. Load Dataset

> **Before running:** make sure you ran notebook 01 to prepare the dataset.
> The expected structure is `data/train/<CLASS>/*.jpg` and `data/val/<CLASS>/*.jpg`.

In [ ]:
train_loader, val_loader = get_dataloaders(
    data_root  = str(DATA_ROOT),
    batch_size = 32,
    num_workers= 0,   # keep 0 on Windows
)

# Quick sanity check
images, labels = next(iter(train_loader))
print(f'Batch shape : {images.shape}')   # (32, 3, 224, 224)
print(f'Label range : {labels.min().item()} – {labels.max().item()}')

Train samples : 7240
Val   samples : 1277
Batch shape : torch.Size([32, 3, 224, 224])
Label range : 0 – 4


## 2. Train MobileNetV2

- Backbone frozen, only head trained
- 15 epochs is enough for a frozen backbone
- Best weights saved to `models/mobilenetv2_best.pt`

In [ ]:
mobilenet = get_model('mobilenetv2', num_classes=NUM_CLASSES)

history_mobilenet = train_model(
    model       = mobilenet,
    train_loader= train_loader,
    val_loader  = val_loader,
    model_name  = 'mobilenetv2',
    save_dir    = str(MODELS_DIR),
    epochs      = 15,
    lr          = 1e-3,
    device      = DEVICE,
)

Training on: cpu


Epoch 01/15 | Train Loss: 0.7189  Acc: 71.99% | Val Loss: 0.5749  Acc: 76.51%
  ✓ Best model saved → C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\v1\models\mobilenetv2_best.pt  (val_acc=76.51%)


Epoch 02/15 | Train Loss: 0.6198  Acc: 75.08% | Val Loss: 0.5622  Acc: 77.53%
  ✓ Best model saved → C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\v1\models\mobilenetv2_best.pt  (val_acc=77.53%)


Epoch 03/15 | Train Loss: 0.5837  Acc: 76.66% | Val Loss: 0.5484  Acc: 78.86%
  ✓ Best model saved → C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\v1\models\mobilenetv2_best.pt  (val_acc=78.86%)


Epoch 04/15 | Train Loss: 0.5744  Acc: 76.70% | Val Loss: 0.5344  Acc: 77.45%


...

Epoch 11/15 | Train Loss: 0.5258  Acc: 78.36% | Val Loss: 0.5018  Acc: 80.66%
  ✓ Best model saved → C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\v1\models\mobilenetv2_best.pt  (val_acc=80.66%)


Epoch 12/15 | Train Loss: 0.5121  Acc: 79.09% | Val Loss: 0.4995  Acc: 81.36%
  ✓ Best model saved → C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\v1\models\mobilenetv2_best.pt  (val_acc=81.36%)


Epoch 13/15 | Train Loss: 0.5180  Acc: 78.55% | Val Loss: 0.5157  Acc: 79.48%


Epoch 14/15 | Train Loss: 0.5226  Acc: 78.69% | Val Loss: 0.5133  Acc: 79.09%


Epoch 15/15 | Train Loss: 0.5078  Acc: 79.21% | Val Loss: 0.5106  Acc: 79.80%

Training complete. Best val accuracy: 81.36%


In [ ]:
plot_training_history(history_mobilenet, model_name='MobileNetV2', save_dir=str(PLOTS_DIR))

Training curves saved → C:\Users\Srusti\Downloads\eclipse-java-2024-09-R-win32-x86_64\project\v1\outputs\plots\MobileNetV2_training_curves.png


## 3. Train Custom CNN

- All weights trained from scratch
- Needs more epochs than MobileNetV2 since there is no pretrained backbone
- Best weights saved to `models/customcnn_best.pt`

In [ ]:
cnn = get_model('customcnn', num_classes=NUM_CLASSES)

history_cnn = train_model(
    model       = cnn,
    train_loader= train_loader,
    val_loader  = val_loader,
    model_name  = 'customcnn',
    save_dir    = str(MODELS_DIR),
    epochs      = 25,
    lr          = 1e-3,
    device      = DEVICE,
)

Training on: cpu


KeyboardInterrupt: 

In [ ]:
plot_training_history(history_cnn, model_name='CustomCNN', save_dir=str(PLOTS_DIR))

## 4. Fine-tune YOLOv8n

YOLOv8 uses its own training loop via the `ultralytics` library.

> **Prerequisite:** The YOLO dataset must be prepared first.
> Run the cell in notebook 02 that generates `data/yolo/dataset.yaml`,
> then populate `data/yolo/train/images/` and `data/yolo/train/labels/`
> with images and YOLO-format `.txt` label files from RDD2022.

The cell below will skip gracefully if the dataset is not ready.

In [ ]:
from ultralytics import YOLO

yaml_path = DATA_ROOT / 'yolo' / 'dataset.yaml'
yolo_train_imgs = DATA_ROOT / 'yolo' / 'train' / 'images'

if not yaml_path.exists() or not any(yolo_train_imgs.glob('*.jpg')):
    print('⚠ YOLO dataset not ready. Skipping YOLOv8 training.')
    print(f'  Expected images at: {yolo_train_imgs}')
    print('  See notebook 02 → Section 4 to generate dataset.yaml')
else:
    yolo = YOLO('yolov8n.pt')
    results = yolo.train(
        data    = str(yaml_path),
        epochs  = 30,
        imgsz   = 640,
        batch   = 16,
        project = str(MODELS_DIR),
        name    = 'yolov8n_rdd2022',
        exist_ok= True,
        verbose = False,
    )
    print(f'✓ YOLOv8 training complete.')
    print(f'  Best weights → {MODELS_DIR}/yolov8n_rdd2022/weights/best.pt')

## 5. Evaluate MobileNetV2

In [ ]:
# Reload best weights
mobilenet_eval = get_model('mobilenetv2', num_classes=NUM_CLASSES)
mobilenet_eval.load_state_dict(torch.load(MODELS_DIR / 'mobilenetv2_best.pt', map_location=DEVICE))

metrics_mobilenet = evaluate_model(
    model      = mobilenet_eval,
    val_loader = val_loader,
    device     = DEVICE,
    save_dir   = str(PLOTS_DIR),
)

## 6. Evaluate Custom CNN

In [ ]:
cnn_eval = get_model('customcnn', num_classes=NUM_CLASSES)
cnn_eval.load_state_dict(torch.load(MODELS_DIR / 'customcnn_best.pt', map_location=DEVICE))

metrics_cnn = evaluate_model(
    model      = cnn_eval,
    val_loader = val_loader,
    device     = DEVICE,
    save_dir   = str(PLOTS_DIR),
)

## 7. Side-by-Side Model Comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.DataFrame([
    {'Model': 'MobileNetV2', **{k: f"{v:.2f}%" for k, v in metrics_mobilenet.items()}},
    {'Model': 'Custom CNN',  **{k: f"{v:.2f}%" for k, v in metrics_cnn.items()}},
])

print('\n── Model Comparison ─────────────────────────────────')
print(comparison.to_string(index=False))

# Bar chart comparison
metric_keys = ['accuracy', 'precision', 'recall', 'f1']
x = range(len(metric_keys))

fig, ax = plt.subplots(figsize=(9, 5))
width = 0.35

mob_vals = [metrics_mobilenet[k] for k in metric_keys]
cnn_vals = [metrics_cnn[k]       for k in metric_keys]

bars1 = ax.bar([i - width/2 for i in x], mob_vals, width, label='MobileNetV2', color='steelblue')
bars2 = ax.bar([i + width/2 for i in x], cnn_vals,  width, label='Custom CNN',  color='coral')

ax.set_xticks(list(x))
ax.set_xticklabels([m.capitalize() for m in metric_keys])
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 110)
ax.set_title('MobileNetV2 vs Custom CNN – Evaluation Metrics')
ax.legend()

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
out = PLOTS_DIR / 'model_comparison.png'
fig.savefig(out, dpi=150)
plt.show()
print(f'✓ Comparison chart saved → {out}')

## 8. Training Curves – Overlay

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Loss
ax1.plot(history_mobilenet['val_loss'], label='MobileNetV2 Val Loss', color='steelblue')
ax1.plot(history_cnn['val_loss'],       label='CustomCNN Val Loss',   color='coral')
ax1.set_title('Validation Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy
ax2.plot(history_mobilenet['val_acc'], label='MobileNetV2 Val Acc', color='steelblue')
ax2.plot(history_cnn['val_acc'],       label='CustomCNN Val Acc',   color='coral')
ax2.set_title('Validation Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend()

plt.suptitle('Training Curves – All Classifiers', fontweight='bold')
plt.tight_layout()
out = PLOTS_DIR / 'all_training_curves.png'
fig.savefig(out, dpi=150)
plt.show()
print(f'✓ Overlay curves saved → {out}')

## ✅ Summary

| Step | Status |
|------|--------|
| MobileNetV2 trained | ✓ |
| Custom CNN trained | ✓ |
| YOLOv8n fine-tuned | ✓ (if YOLO dataset ready) |
| Accuracy / Precision / Recall / F1 computed | ✓ |
| Confusion matrix saved | ✓ |
| Training curves saved | ✓ |
| Model comparison chart saved | ✓ |

**Next:** Open `04_final_demo.ipynb` for the end-to-end inference demo.